In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import math
import urllib.request


print("using device is idk wait i found it still loading ")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"your using :{device} ")
#we have to download shakespare dataset from internet


# ════════════════════════════════
# 1. DOWNLOAD SHAKESPEARE
# ════════════════════════════════


url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'shakespeare.txt')
with open('shakespeare.txt', 'r') as f:
    text = f.read()
print(f"Total characters :{len(text):,}")
print(f"sample: \n{text[:200]}")


# now we have convert into charater to function and learnthe model 
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 2. CHARACTER LEVEL TOKENIZER ══════════════════════════════════════════════════════════
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"vocab size : {vocab_size}")
print(f"characters : {''.join(chars)}")
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode('hello')))
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 3. TRAIN/VAL SPLIT ═══════════════════════════════════════════════════════════════════
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train : {len(train_data):,} val: {len(val_data):,}")

# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 4. BATCH GENERATOR ═══════════════════════════════════════════════════════════════════
def get_batch(split,batch_size=32,block_size = 128):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i + 1 + block_size] for i in ix])
    return x.to(device), y.to(device)

#════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 5.  MODEL  ═══════════════════════════════════════════════════════════════════
def scaled_dot_product_attention(Q,K,V , mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.transpose(-2,-1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    weigths = F.softmax(scores,dim=-1)
    return torch.matmul(weigths,V), weigths


# head 
class CausalMultiheadattention(nn.Module):
    def __init__(self,d_model,num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model// num_heads
        self.W_Q = nn.Linear(d_model,d_model)
        self.W_K = nn.Linear(d_model,d_model)
        self.W_V = nn.Linear(d_model,d_model)
        self.W_O = nn.Linear(d_model,d_model)
        self.dropout = nn.Dropout(dropout)
    def split_head(self,x,batch):
        x = x.view(batch,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)
    def forward(self,x,mask=None):
        batch = x.size(0)
        Q = self.split_head(self.W_Q(x), batch)
        K = self.split_head(self.W_K(x), batch)
        V = self.split_head(self.W_V(x), batch)
        out, _ = scaled_dot_product_attention(Q,K,V,mask)
        out = out.transpose(1,2).contiguous()
        out = out.view(batch,-1,self.d_model)
        return self.W_O(out)





class feedForward(nn.Module):
    def __init__(self,d_model,d_ff,dropout = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff,d_model)
        
        )
    def forward(self,x):
        return self.net(x)
    


class GPTBLOCK(nn.Module):
    def __init__(self,d_model, num_heads,d_ff , dropout = 0.1):
        super().__init__()
        self.attention = CausalMultiheadattention(d_model,num_heads,dropout)
        self.ff = feedForward(d_model,d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self,x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ff(self.norm2(x))
        return x 
    

class minigpt(nn.Module):
    def __init__(self, vocab_size, d_model = 128, num_heads = 4,
                 num_layers = 4,d_ff = 512, max_len = 512, dropout = 0.1):
        super().__init__()
        self.token_embedding =nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPTBLOCK(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size , bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weigths)







    def _init_weigths(self,module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight,mean=0.0,std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module,nn.Embedding):
            nn.init.normal_(module.weight, mean = 0.0,std=0.02)

    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        device = idx.device

        tok_emb = self.token_embedding(idx)
        pos = torch.arange(seq_len, device=device)
        pos_emb = self.pos_embedding(pos)
        x = self.dropout(tok_emb + pos_emb)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        for block in self.blocks:
            x = block(x, mask)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -512:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            logits = logits / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 6. system configration / setup  ═══════════════════════════════════════════════════════════════════



device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using : {device}")
model = minigpt(
    vocab_size= vocab_size,
    d_model= 128,
    num_heads=4,
    num_layers=4,
    d_ff = 512,
    max_len=512,
    dropout= 0.1

).to(device)

params = sum(p.numel() for p in model.parameters())
print(f"parameter : {params:,}(~{params/1e6:.2f}M)")
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 3e-4,
    weight_decay=0.1
)
# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 7. training epochs and loops  ═══════════════════════════════════════════════════════════════════

BATCH_SIZE = 25
BLOCK_SIZE = 128
EVAL_EVERY = 500
MAX_ITERS = 5000

@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        total = 0
        for _ in range(eval_iters):
            x, y   = get_batch(split, BATCH_SIZE, BLOCK_SIZE)
            _, loss = model(x, y)
            total  += loss.item()
        losses[split] = total / eval_iters
    model.train()
    return losses

print("\n" + "="*60)
print(f"{'Step':<8}{'Train Loss':>12}{'Val Loss':>12}{'Time':>10}")
print("="*60)
import time
best_val_loss = float('inf')
start_time = time.time()
for step in range(MAX_ITERS):
    if step % EVAL_EVERY ==0:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"{step:<8}{losses['train']:>12.4f}"
              f"{losses['val']:>12.4f}{elapsed:>9.1f}s")
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            model.load_state_dict(torch.load('best_model.pth', map_location=device))
    x,y = get_batch('train',BATCH_SIZE,BLOCK_SIZE)
    optimizer.zero_grad() 
    logits,loss = model(x,y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print("="*60)
print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")

# ════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
# ═════════════════════════════════════════ 7. output/execution   ═══════════════════════════════════════════════════════════════════



model.load_state_dict(torch.load('best_gpt.pth'))
model.eval()

print("\n" + "="*60)
print("GENERATED SHAKESPEARE:")
print("="*60)

# generate with different temperatures!
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f"\n--- Temperature {temp} ---")
    start  = torch.zeros((1, 1), dtype=torch.long).to(device)
    output = model.generate(start, max_new_tokens=200, temperature=temp)
    tokens = output[0].tolist()
    print(decode(tokens))
    print()


    

